<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Deep-Learning-with-Python/blob/main/scripts/chapter09_convnet_architecture_patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install keras keras-hub --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.32.0 which is incompatible.


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## ConvNet architecture patterns

### Modularity, hierarchy, and reuse

### Residual connections

In [ ]:
import keras
from keras import layers

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
residual = layers.Conv2D(64, 1)(residual)
x = layers.add([x, residual])

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)
residual = layers.Conv2D(64, 1, strides=2)(residual)
x = layers.add([x, residual])

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Rescaling(1.0 / 255)(inputs)

def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)
    x = layers.add([x, residual])
    return x

x = residual_block(x, filters=32, pooling=True)
x = residual_block(x, filters=64, pooling=True)
x = residual_block(x, filters=128, pooling=False)

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

### Batch normalization

### Depthwise separable convolutions

### Putting it together: A mini Xception-like model

In [ ]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import zipfile

download_path = kagglehub.competition_download("dogs-vs-cats")

with zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:
    zip_ref.extractall(".")

100%|██████████| 812M/812M [00:07<00:00, 119MB/s]

Extracting files...


In [ ]:
import os, shutil, pathlib
from keras.utils import image_dataset_from_directory

original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("dogs_vs_cats_small")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)

batch_size = 64
image_size = (180, 180)
train_dataset = image_dataset_from_directory(
    new_base_dir / "train",
    image_size=image_size,
    batch_size=batch_size,
)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation",
    image_size=image_size,
    batch_size=batch_size,
)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test",
    image_size=image_size,
    batch_size=batch_size,
)

Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [ ]:
import tensorflow as tf
from keras import layers

data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
]

def data_augmentation(images, targets):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images, targets

augmented_train_dataset = train_dataset.map(
    data_augmentation, num_parallel_calls=8
)
augmented_train_dataset = augmented_train_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
import keras

inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1.0 / 255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=5, use_bias=False)(x)

for size in [32, 64, 128, 256, 512]:
    residual = x

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    residual = layers.Conv2D(
        size, 1, strides=2, padding="same", use_bias=False
    )(residual)
    x = layers.add([x, residual])

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
history = model.fit(
    augmented_train_dataset,
    epochs=100,
    validation_data=validation_dataset,
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 91s 2s/step - accuracy: 0.5640 - loss: 0.6792 - val_accuracy: 0.5000 - val_loss: 0.6952
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 426ms/step - accuracy: 0.6310 - loss: 0.6427 - val_accuracy: 0.5000 - val_loss: 0.6938
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 617ms/step - accuracy: 0.6365 - loss: 0.6303 - val_accuracy: 0.5040 - val_loss: 0.6926
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 443ms/step - accuracy: 0.6755 - loss: 0.6085 - val_accuracy: 0.4990 - val_loss: 0.6940
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 398ms/step - accuracy: 0.6980 - loss: 0.5834 - val_accuracy: 0.5000 - val_loss: 0.6964
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 394ms/step - accuracy: 0.6910 - loss: 0.5930 - val_accuracy: 0.5000 - val_loss: 0.7166
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 408ms/step - accuracy: 0.7160 - loss: 0.5700 - val_accuracy: 0.5000 - val_loss: 0.7107
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 413ms/step - accuracy: 0.7295 - loss: 0.5400 - val

### Beyond convolution: Vision Transformers

### **Why Fully Covolutional Network**

In [ ]:
# @title **1. Convnet Fails!**

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# =========================================================
# 1. ConvNet 모델 생성 함수 (Flatten 포함)
# =========================================================
def build_convnet(input_shape):
    inputs = keras.Input(shape=input_shape)

    # Feature Extraction (합성곱 및 풀링)
    x = layers.Conv2D(32, kernel_size=3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Conv2D(64, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)

    # 💥 문제의 Flatten 지점
    x = layers.Flatten()(x)

    # Classification (Dense 레이어)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(4, activation="softmax")(x)  # 4개 클래스 분류

    return keras.Model(inputs=inputs, outputs=outputs)


# =========================================================
# 2. Case 1: 64x64 입력 데이터로 모델 학습 및 가중치 저장
# =========================================================
print("--- [Case 1] 64x64 크기 입력으로 모델 빌드 및 가중치 저장 ---")
model_64 = build_convnet(input_shape=(64, 64, 1))

# 64x64 입력 시 Flatten 출력 차원 확인: (64/4) * (64/4) * 64 = 16 * 16 * 64 = 16,384개
model_64.summary()

# 가중치 파일로 저장
model_64.save_weights("convnet_64x64.weights.h5")
print("64x64 가중치 저장 완료!\n")


# =========================================================
# 3. Case 2: 32x32 입력을 처리하기 위해 모델을 불러와 실행 시도 (Fail 발생)
# =========================================================
print("--- [Case 2] 32x32 크기 입력을 처리하기 위해 가중치 불러오기 시도 ---")
model_32 = build_convnet(input_shape=(32, 32, 1))

# 32x32 입력 시 Flatten 출력 차원: (32/4) * (32/4) * 64 = 8 * 8 * 64 = 4,096개

try:
    # 💥 저장된 64x64용 가중치(16,384차원용)를 32x32 모델(4,096차원용)에 로드 시도
    model_32.load_weights("convnet_64x64.weights.h5")

    # 가상의 32x32 테스트 이미지 1장 생성 후 예측 시도
    mock_input_32 = tf.random.normal([1, 32, 32, 1])
    prediction = model_32(mock_input_32)
    print("예측 성공:", prediction.shape)

except Exception as e:
    print("\n❌ [FAIL] 차원 불일치(Shape Mismatch) 에러가 발생했습니다!")
    print(f"에러 메시지: {e}")

--- [Case 1] 64x64 크기 입력으로 모델 빌드 및 가중치 저장 ---


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_35 (Conv2D)              │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_36 (Conv2D)              │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_24 (MaxPooling2D) │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,116,612 (8.07 MB)

 Trainable params: 2,116,612 (8.07 MB)

 Non-trainable params: 0 (0.00 B)

64x64 가중치 저장 완료!

--- [Case 2] 32x32 크기 입력을 처리하기 위해 가중치 불러오기 시도 ---

❌ [FAIL] 차원 불일치(Shape Mismatch) 에러가 발생했습니다!
에러 메시지: A total of 1 objects could not be loaded. Example error message for object <Dense name=dense_21, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(4096, 128), Received: value.shape=(16384, 128). Target variable: <Variable path=dense_21/kernel, shape=(4096, 128), dtype=float32, value=[[-0.02713202 -0.03131886  0.01865802 ...  0.02302274 -0.00172336
  -0.01185644]
 [-0.00574154 -0.00698463 -0.0338694  ...  0.02797553  0.02860507
   0.01287301]
 [-0.00874008 -0.02106974 -0.02328406 ...  0.00457723  0.02073152
   0.01139321]
 ...
 [ 0.02975319 -0.03442844  0.00220653 ... -0.02971879  0.00352192
   0.03565065]
 [-0.01886655  0.0059818  -0.0369659  ... -0.0074115   0.03618341
  -0.00599827]
 [-0.01489915 -0.02382424  0.02023454 ... -0.0257661   0.01651504
  -0.02757464]]>

List of object

In [ ]:
# @title **2. FCN(Fully Convolutional Network) succeed!**

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_fcn_model():
    # 💡 핵심: 가로, 세로 크기를 None으로 지정하여 가변 입력 크기를 허용합니다.
    inputs = keras.Input(shape=(None, None, 1))

    # 특징 추출부 (LeNet-5 구조 그대로 유지)
    x = layers.Conv2D(filters=6, kernel_size=5, padding="valid", activation="relu")(inputs)
    x = layers.MaxPooling2D(pool_size=2, strides=2)(x)
    x = layers.Conv2D(filters=16, kernel_size=5, padding="valid", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=2, strides=2)(x)

    # ⭕ [FCN 핵심 변환]: Dense 층들을 전부 Conv2D로 대체합니다.
    x = layers.Conv2D(filters=120, kernel_size=5, padding="valid", activation="relu")(x)
    x = layers.Conv2D(filters=84, kernel_size=1, activation="relu")(x)
    outputs = layers.Conv2D(filters=10, kernel_size=1, activation="softmax")(x)

    return keras.Model(inputs=inputs, outputs=outputs)


# 1. 단일 FCN 모델 생성 (동일한 가중치/파라미터 사용)
fcn_model = build_fcn_model()

print("=== FCN 다양한 입력 크기 테스트 ===")

# 2. 서로 다른 3가지 크기의 입력 생성
input_32 = tf.random.normal([1, 32, 32, 1])   # Case 1: 32 x 32
input_64 = tf.random.normal([1, 64, 64, 1])   # Case 2: 64 x 64
input_128 = tf.random.normal([1, 128, 128, 1]) # Case 3: 128 x 128

# 3. 각 입력별 통과 및 최종 출력 Spatial Grid Shape 확인
pred_32 = fcn_model(input_32)
pred_64 = fcn_model(input_64)
pred_128 = fcn_model(input_128)

print(f"👉 32x32   입력 시 최종 출력 Shape: {pred_32.shape}")
print(f"👉 64x64   입력 시 최종 출력 Shape: {pred_64.shape}")
print(f"👉 128x128 입력 시 최종 출력 Shape: {pred_128.shape}")

=== FCN 다양한 입력 크기 테스트 ===
👉 32x32   입력 시 최종 출력 Shape: (1, 1, 1, 10)
👉 64x64   입력 시 최종 출력 Shape: (1, 9, 9, 10)
👉 128x128 입력 시 최종 출력 Shape: (1, 25, 25, 10)
